In [ ]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt


# -----------------------------
# Reusable helper functions
# -----------------------------
def clean_column_names(dataframe):
    """Return a copy with leading/trailing whitespace removed from column names."""
    dataframe = dataframe.copy()
    dataframe.columns = dataframe.columns.astype(str).str.strip()
    return dataframe


def first_existing_column(dataframe, candidates, required=True, purpose="column"):
    """Find the first column from candidates that exists in dataframe."""
    for col in candidates:
        if col in dataframe.columns:
            return col
    if required:
        raise ValueError(
            f"Could not find a {purpose}. Tried: {candidates}. "
            f"Available columns are: {list(dataframe.columns)}"
        )
    return None


def clean_group_labels(series):
    """Normalize labels like 'A - 2' or 'A-2' into 'A2'."""
    return (
        series
        .astype(str)
        .str.upper()
        .str.replace(" ", "", regex=False)
        .str.replace("-", "", regex=False)
    )


def safe_divide(numerator, denominator, pseudocount=0):
    """Divide safely and return NaN when denominator is zero/missing."""
    denom = pd.to_numeric(denominator, errors="coerce") + pseudocount
    denom = denom.replace(0, np.nan)
    return pd.to_numeric(numerator, errors="coerce") / denom


def add_feature_if_possible(dataframe, new_col, numerator_col, denominator_col=None, operation="copy", pseudocount=0):
    """Add an engineered feature only if its required source columns exist."""
    dataframe = dataframe.copy()
    if operation == "copy" and numerator_col in dataframe.columns:
        dataframe[new_col] = pd.to_numeric(dataframe[numerator_col], errors="coerce")
    elif operation == "ratio" and numerator_col in dataframe.columns and denominator_col in dataframe.columns:
        dataframe[new_col] = safe_divide(dataframe[numerator_col], dataframe[denominator_col], pseudocount=pseudocount)
    elif operation == "sum" and numerator_col in dataframe.columns and denominator_col in dataframe.columns:
        dataframe[new_col] = (
            pd.to_numeric(dataframe[numerator_col], errors="coerce") +
            pd.to_numeric(dataframe[denominator_col], errors="coerce")
        )
    return dataframe


def choose_features(dataframe, requested_features=None, exclude_columns=None):
    """Use requested features when provided; otherwise use all numeric columns except metadata/output columns."""
    exclude_columns = set(exclude_columns or [])

    if requested_features:
        features_found = [col for col in requested_features if col in dataframe.columns]
        missing = [col for col in requested_features if col not in dataframe.columns]
        if missing:
            print("Requested features not found and skipped:")
            for col in missing:
                print(f" - {col}")
        return features_found

    numeric_cols = dataframe.select_dtypes(include=[np.number]).columns.tolist()
    return [col for col in numeric_cols if col not in exclude_columns]


def build_feature_matrix(dataframe, features, fill_strategy="median"):
    """Convert selected features to numeric, drop unusable columns, and fill missing values."""
    if not features:
        raise ValueError("No usable numeric features were found. Add feature names to REQUESTED_FEATURES or check your input data.")

    X = dataframe[features].copy() #X contains only the numeric features that the model will use.

    # We will need to convert these to numeric and handle any missing values before feeding them into the GMM:
    # Convert to numeric
    X = X.apply(pd.to_numeric, errors="coerce")

    # Drop features that are completely missing after numeric conversion.
    all_missing = X.columns[X.isna().all()].tolist()
    if all_missing:
        print("Features with all missing/non-numeric values were dropped:")
        for col in all_missing:
            print(f" - {col}")
        X = X.drop(columns=all_missing)

    # Drop features with no variance; they do not help distance/variance-based models.
    constant_cols = [col for col in X.columns if X[col].nunique(dropna=True) <= 1]
    if constant_cols:
        print("Features with no variance were dropped:")
        for col in constant_cols:
            print(f" - {col}")
        X = X.drop(columns=constant_cols)

    if X.empty:
        raise ValueError("After cleaning, no usable numeric features remain.")

    # -and then we can fill missing values with the median of each column (not really a problem in the dataset but good practice):
    # Fill missing values with column medians
    if fill_strategy == "mean":
        X = X.fillna(X.mean()) # This could also hypothetically be the mean:
    else:
        X = X.fillna(X.median()) # This could also hypothetically be the mean:
    # X = X.fillna(X.mean())

    return X, X.columns.tolist()


### Simple Anchor-Based GMM

#### Goal

Use **A2** and **B2** as anchor wells to fit a two-group Gaussian Mixture Model, then classify every row as either **A2-like** or **B2-like**.

#### Input

`simple_universal_live_dead_combined.csv`

#### Output

Classification results assigning each row to one of the following groups:

- `A2-like`
- `B2-like`

#### Ideas for the Future

- Transition to lightly supervised methods, such as **label spreading**, instead of strictly unsupervised methods, such as **GMM**.  
  Reference: [Semi-Supervised Learning](https://www.geeksforgeeks.org/machine-learning/ml-semi-supervised-learning/)
- Use more than two anchor wells.
- Try a different clustering method.
- Limit the number of features and/or use dimensionality reduction, such as **PCA**.
- Add more features, such as morphological features.

In [ ]:
# -----------------------------
# 1. Load data
# -----------------------------
# Change INPUT_FILE to reuse this notebook with a different CSV.
INPUT_FILE = "simple_universal_live_dead_combined.csv"
OUTPUT_PREFIX = Path(INPUT_FILE).stem

# Optional: set this if you want outputs saved somewhere specific.
OUTPUT_DIR = Path(".")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(INPUT_FILE)

# Clean column names
df = clean_column_names(df) #This removes any leading/trailing whitespace from column names

print(f"Loaded {len(df):,} rows and {len(df.columns):,} columns from {INPUT_FILE}")


In [ ]:
df.head() # This displays the first few rows of the dataframe to verify that it loaded correctly and to inspect the structure of the data.


In [ ]:
df.describe(include="all") # This provides summary statistics for each column, which can help us understand the distribution of the data and identify any potential issues (e.g., missing values, outliers).


In [ ]:
# Unique groups in the time_hr column
TIME_COLUMN_CANDIDATES = ["time_hr", "Time", "time", "hour", "hours"]
time_col = first_existing_column(df, TIME_COLUMN_CANDIDATES, required=False, purpose="time column")

if time_col:
    print(f"Unique groups in {time_col} column:", df[time_col].dropna().unique())
else:
    print("No time column found. This is okay if your dataset does not use time filters.")


In [ ]:
# Unique groups in the  temperature_c column
TEMPERATURE_COLUMN_CANDIDATES = ["temperature_c", "Temperature", "temperature", "temp_c", "temp"]
temperature_col = first_existing_column(df, TEMPERATURE_COLUMN_CANDIDATES, required=False, purpose="temperature column")

if temperature_col:
    print(f"Unique groups in {temperature_col} column:", df[temperature_col].dropna().unique())
else:
    print("No temperature column found. This is okay if your dataset does not use temperature filters.")


In [ ]:
# Optional: inspect all columns when adapting to a new dataset.
print("Columns in this dataset:")
for col in df.columns:
    print(" -", col)


# Change Stuff Here!

In [ ]:
# Filter to look at only the 48hr time point
# df=df[df["time_hr"] == 48]

# Generalized filters:
# - Leave a value as None to skip that filter.
# - Add or remove keys to filter on any column in any dataset.
FILTERS = {
    # "time_hr": [48],
    "time_hr": [48, 72],
    # "temperature_c": [4, 22, 37],
    "temperature_c": [4, 22],
}

# Keep only filters whose columns exist in this dataset.
for filter_col, allowed_values in FILTERS.items():
    if allowed_values is None:
        continue
    if filter_col not in df.columns:
        print(f"Skipping filter because column was not found: {filter_col}")
        continue
    df = df[df[filter_col].isin(allowed_values)].copy()

print(f"Rows after filtering: {len(df):,}")


In [ ]:
# -----------------------------
# 2. Clean WELL LABEL
# -----------------------------
# This converts labels like "A - 2" or "A-2" into "A2"
GROUP_COLUMN_CANDIDATES = ["WELL LABEL", "well", "well_label", "Well", "Sample", "sample", "group", "Group"]
group_col = first_existing_column(df, GROUP_COLUMN_CANDIDATES, required=True, purpose="group / anchor label column")

# Use WELL_CLEAN as a generic cleaned grouping column, even when your input column is not literally a well label.
df["WELL_CLEAN"] = clean_group_labels(df[group_col])
# What this does is ensure that all well labels are in a consistent format, which is crucial for accurate grouping and analysis later on.

print(f"Using '{group_col}' as the anchor/group column.")
print("Cleaned group labels:", sorted(df["WELL_CLEAN"].dropna().unique()))


In [ ]:
# Drop some WELL CLEAN points to simplify the dataset and make it easier to visualize
# wells_to_drop = [
#                  "A5", "A8", "A11", 
#                 "B5", "B8", "B11", 
#                  "C2", "C5", "C8", "C11", 
#                  "D2", "D5", "D8", "D11",
#                    "E2", "E5", "E8", "E11", 
#                    "F2", "F5", "F8", "F11", 
#                    "G2", "G5", "G8", "G11", 
#                    "H2", "H5", "H8", "H11"

#                    ]
wells_to_drop = [
                 "D2","G5","F8",
#  "D5", "D8", "D11",
#                    "E2", "E5", "E8", "E11", 
                   "F2", "F5",  "F11", 
                   "G2",  "G8", "G11", 
                   "H2", "H5", "H8", "H11"

                   ]

# Set wells_to_drop = [] when reusing this notebook and you do not want to drop any groups.
wells_to_drop = [clean_group_labels(pd.Series([w])).iloc[0] for w in wells_to_drop]
wells_to_drop_present = [w for w in wells_to_drop if w in set(df["WELL_CLEAN"])]
if wells_to_drop_present:
    print("Dropping groups:", wells_to_drop_present)
df = df[~df["WELL_CLEAN"].isin(wells_to_drop)].copy()
print(f"Rows after dropping selected groups: {len(df):,}")


In [ ]:
# -----------------------------
# 3. Choose anchor wells
# -----------------------------
# Change these values to any two anchor groups in your dataset.
anchor_A = "A2" # Media Group
anchor_B = "B2" # Lysis Group

# Optional custom label names. These default to f"{anchor}-like" if left as None.
anchor_A_label = None
anchor_B_label = None

anchor_A_clean = clean_group_labels(pd.Series([anchor_A])).iloc[0]
anchor_B_clean = clean_group_labels(pd.Series([anchor_B])).iloc[0]
anchor_A_label = anchor_A_label or f"{anchor_A_clean}-like"
anchor_B_label = anchor_B_label or f"{anchor_B_clean}-like"

# We will use these anchors to define the "Live" and "Dead" clusters in the GMM step.
print("Anchor A:", anchor_A_clean, "=>", anchor_A_label)
print("Anchor B:", anchor_B_clean, "=>", anchor_B_label)


In [ ]:
#### EXPERIMENTAL:

# Optional engineered features. These are added only when the required columns exist,
# so the notebook can be reused with datasets that do not have these exact Live/Dead columns.
df = add_feature_if_possible(df, "Total Count", "Live Live Count", "Dead Dead Count", operation="sum")

if "Total Count" in df.columns and "Live Live Count" in df.columns:
    df["Live Fraction"] = safe_divide(df["Live Live Count"], df["Total Count"])
if "Total Count" in df.columns and "Dead Dead Count" in df.columns:
    df["Dead Fraction"] = safe_divide(df["Dead Dead Count"], df["Total Count"])

if "Live Live Count" in df.columns and "Dead Dead Count" in df.columns:
    df["Live Dead Count Ratio"] = safe_divide(df["Live Live Count"], df["Dead Dead Count"], pseudocount=1)

if "Live Live Volume_MEAN" in df.columns and "Dead Dead Volume_MEAN" in df.columns:
    df["Live Dead Volume Ratio"] = (
        safe_divide(df["Live Live Volume_MEAN"], df["Dead Dead Volume_MEAN"], pseudocount=1e-9)
    )

if "Live Live Mean Intensity 1 : FITC - FITC_MEAN" in df.columns and "Dead Dead Mean Intensity 2 : TX RED - TX RED_MEAN" in df.columns:
    df["Live Dead Intensity Ratio"] = (
        safe_divide(
            df["Live Live Mean Intensity 1 : FITC - FITC_MEAN"],
            df["Dead Dead Mean Intensity 2 : TX RED - TX RED_MEAN"],
            pseudocount=1e-9
        )
    )

print("Engineered features present:")
for col in ["Total Count", "Live Fraction", "Dead Fraction", "Live Dead Count Ratio", "Live Dead Volume Ratio", "Live Dead Intensity Ratio"]:
    if col in df.columns:
        print(" -", col)


In [ ]:
# -----------------------------
# 4. Choose features
# WE CAN ADD MORE FEATURES IF WE WANT BASED ON THE DATASET. THIS IS JUST A STARTING POINT.
# -----------------------------
# Option 1: Provide a feature list here.
# Option 2: Set REQUESTED_FEATURES = None to automatically use all numeric columns.
REQUESTED_FEATURES = [
    "Live Live Volume_MEAN",
    "Live Live Diameter_MEAN",
    "Live Live Surface Area_MEAN",
    "Live Live Compactness_MEAN",
    "Live Live Sphericity_MEAN",
    "Live Live Count",
    "Live Live Mean Intensity 1 : FITC - FITC_MEAN",
    "Live Live Total Intensity 1 : FITC - FITC_MEAN",

    "Dead Dead Volume_MEAN",
    "Dead Dead Diameter_MEAN",
    "Dead Dead Surface Area_MEAN",
    "Dead Dead Compactness_MEAN",
    "Dead Dead Sphericity_MEAN",
    "Dead Dead Count",
    "Dead Dead Mean Intensity 2 : TX RED - TX RED_MEAN",
    "Dead Dead Total Intensity 2 : TX RED - TX RED_MEAN",


    "Total Count",
    "Live Fraction",
    "Dead Fraction",
    "Live Dead Count Ratio",
    "Live Dead Volume Ratio",
    "Live Dead Intensity Ratio",
]

# To make this fully automatic for a different dataset, uncomment this line:
# REQUESTED_FEATURES = None

# Columns that should not become model features when REQUESTED_FEATURES = None.
EXCLUDE_FROM_AUTO_FEATURES = [
    group_col,
    "WELL_CLEAN",
    "GMM_Component",
    "GMM_Group",
    "Probability_A_like",
    "Probability_B_like",
    "PCA1",
    "PCA2",
]

features = choose_features(df, requested_features=REQUESTED_FEATURES, exclude_columns=EXCLUDE_FROM_AUTO_FEATURES)

# This is the list of columns the GMM will use to decide similarity.
# We include both "Live" and "Dead" features to give the model a comprehensive view of the cell characteristics in the wells. 
# The GMM will then cluster the wells based on these features, ideally grouping the "Live" wells together and the "Dead" wells together, with the anchor wells helping to define these clusters.

print(f"Using {len(features)} feature(s):")
for col in features:
    print(" -", col)


In [ ]:
# features = [
#     "Live Live Count",
#     "Dead Dead Count",
#     "Live Live Volume_MEAN",
#     "Dead Dead Volume_MEAN",
#     "Live Live Mean Intensity 1 : FITC - FITC_MEAN",
#     "Dead Dead Mean Intensity 2 : TX RED - TX RED_MEAN",
# ]

# Another reusable pattern:
# features = [col for col in df.columns if "Intensity" in col]
# features = [col for col in df.columns if col.endswith("_MEAN")]


In [ ]:
# Keep only feature columns that actually exist
features = [col for col in features if col in df.columns]
# This makes sure we don't run into errors later if some expected columns are missing from the dataset.

print(f"Feature count after checking columns: {len(features)}")


In [ ]:
# -----------------------------
# 5. Prepare feature matrix
# -----------------------------
X, features = build_feature_matrix(df, features, fill_strategy="median")
print(f"Final feature matrix shape: {X.shape}")


In [ ]:
# -----------------------------
# 6. Scale features- This is ESSENTIAL for GMM because it is distance/variance-based.
# -----------------------------


scaler = StandardScaler() # This will standardize the features to have mean=0 and std=1, which helps the GMM perform better.
X_scaled = scaler.fit_transform(X) # This is the final scaled feature matrix that we will feed into the GMM.


In [ ]:
# -----------------------------
# 7. Create anchor-based starting means by developing the average profile of the anchor wells in the scaled feature space. 
# -----------------------------

# This will help the GMM converge to meaningful clusters that correspond to "Live" and "Dead" wells as defined by our anchors which are the starting weights for the GMM.
# We will calculate the mean feature values for the wells corresponding to anchor_A and anchor_B, and use these as the initial means for the GMM.

A_mask = (df["WELL_CLEAN"] == anchor_A_clean).to_numpy() # This creates a boolean mask that is True for rows where the WELL_CLEAN column matches anchor_A (e.g., "A2") and False otherwise.
B_mask = (df["WELL_CLEAN"] == anchor_B_clean).to_numpy() # while this does the same for anchor_B (e.g., "B2"). 

# We will use these masks to select the rows corresponding to our anchor wells and calculate their mean feature values.


# We also add error handling to check if the anchor wells are actually present in the dataset. If not, we raise a ValueError with a clear message and stops the code.
if A_mask.sum() == 0:
    available = sorted(df["WELL_CLEAN"].dropna().unique())
    raise ValueError(f"No rows found for anchor well {anchor_A_clean}. Available cleaned groups: {available}")

if B_mask.sum() == 0:
    available = sorted(df["WELL_CLEAN"].dropna().unique())
    raise ValueError(f"No rows found for anchor well {anchor_B_clean}. Available cleaned groups: {available}")

# This calculates the mean feature values for the rows corresponding to anchor_A in the scaled feature space. 
A_mean = X_scaled[A_mask].mean(axis=0) # This will serve as the initial mean for the "Live" cluster in the GMM.

# This calculates the mean feature values for the rows corresponding to anchor_B in the scaled feature space. 
B_mean = X_scaled[B_mask].mean(axis=0) # This will serve as the initial mean for the "Dead" cluster in the GMM.

initial_means = np.vstack([A_mean, B_mean]) 
# This creates a 2D array where the first row is the mean feature values for anchor_A and the second row is the mean feature values for anchor_B which we will use as the initial means for the GMM clusters.


In [ ]:
# Optional sanity check: see how many rows are being used for each anchor.
print(f"Rows for {anchor_A_clean}: {A_mask.sum():,}")
print(f"Rows for {anchor_B_clean}: {B_mask.sum():,}")


In [ ]:
# -----------------------------
# 8. Fit simple 2-component GMM
# -----------------------------
gmm = GaussianMixture(
    n_components=2, # We want to find 2 clusters corresponding to "Live" and "Dead" wells.
    means_init=initial_means, # This tells the GMM to start with the means we calculated from our anchor wells in the block above, which should help it find meaningful clusters corresponding to "Live" and "Dead" wells.
    random_state=42,
    covariance_type="full",
    reg_covar=1e-6, # Small regularization helps the model handle highly similar or small datasets.
)

gmm.fit(X_scaled) # This fits the GMM to our scaled feature data, using the anchor-based initial means to guide the clustering process.


In [ ]:
# -----------------------------
# 9. Predict groups
# -----------------------------
component = gmm.predict(X_scaled) # This assigns each well to one of the two clusters (0 or 1) based on the fitted GMM model. We will later determine which cluster corresponds to "Live" and which corresponds to "Dead" based on the anchor wells.
probabilities = gmm.predict_proba(X_scaled) # This gives us the probability that each well belongs to each of the two clusters, which can be useful for assessing the confidence of the cluster assignments and for downstream analysis.


In [ ]:
# -----------------------------
# 10. Decide which component is A2-like and B2-like
# -----------------------------
# We compare each learned GMM component to the A2 and B2 anchor means.


# This calculates the Euclidean distance from each GMM component mean to the A_mean (anchor_A mean) in the feature space. 
distance_to_A = np.linalg.norm(gmm.means_ - A_mean, axis=1)
# The component with the smaller distance to A_mean is likely the "Live" cluster.


# This calculates the Euclidean distance from each GMM component mean to the B_mean (anchor_B mean) in the feature space. 
distance_to_B = np.linalg.norm(gmm.means_ - B_mean, axis=1)
# The component with the smaller distance to B_mean is likely the "Dead" cluster.



# We determine which GMM component is closer to the A_mean (anchor_A) and which is closer to the B_mean (anchor_B) to assign labels to the clusters.
A_component = int(np.argmin(distance_to_A))
B_component = int(np.argmin(distance_to_B))

# If this accidentally assigns the same idea badly, this still keeps two groups.
if A_component == B_component:
    B_component = 1 - A_component

component_to_label = {
    A_component: anchor_A_label,
    B_component: anchor_B_label
}

print("Component labels:", component_to_label)


In [ ]:
# -----------------------------
# 11. Add results to dataframe
# -----------------------------
df["GMM_Component"] = component # This adds a new column to the original dataframe that indicates which GMM component (0 or 1) each well was assigned to based on the clustering.

df["GMM_Group"] = df["GMM_Component"].map(component_to_label) # This maps the GMM component numbers (0 or 1) to the corresponding labels ("A2-like" or "B2-like") based on which component is closer to the A_mean and B_mean.

df[f"Probability_{anchor_A_clean}_like"] = probabilities[:, A_component] # This adds a new column to the dataframe that gives the probability that each well belongs to the A2-like cluster, which can be useful for assessing the confidence of the cluster assignments and for downstream analysis.

df[f"Probability_{anchor_B_clean}_like"] = probabilities[:, B_component] # This adds a new column to the dataframe that gives the probability that each well belongs to the B2-like cluster, which can be useful for assessing the confidence of the cluster assignments and for downstream analysis.

# Backward-compatible names for your original plotting/summary cells.
df["Probability_A2_like"] = df[f"Probability_{anchor_A_clean}_like"]
df["Probability_B2_like"] = df[f"Probability_{anchor_B_clean}_like"]

# Force anchor rows to keep their anchor labels
df.loc[A_mask, "GMM_Group"] = f"{anchor_A_clean}-anchor" # This ensures that the rows corresponding to anchor_A (e.g., "A2") are labeled as "A2-anchor" in the GMM_Group column, which serves as a sanity check and helps us interpret the clusters correctly.

df.loc[B_mask, "GMM_Group"] = f"{anchor_B_clean}-anchor" # This ensures that the rows corresponding to anchor_B (e.g., "B2") are labeled as "B2-anchor" in the GMM_Group column, which serves as a sanity check and helps us interpret the clusters correctly.


In [ ]:
# -----------------------------
# PCA for visualization only- PCA takes the 16-dimensional measurement space and compresses it into two new summary axes (PCA1 and PCA2) that capture the most variance in the data. This allows us to visualize the clusters in a 2D plot while still retaining as much of the original information as possible.
# -----------------------------
pca = PCA(n_components=2) 
X_pca = pca.fit_transform(X_scaled)

# Basically this is saying that we take all the scaled Live/Dead feature columns and reduce them to 2 dimensions.

df["PCA1"] = X_pca[:, 0]
df["PCA2"] = X_pca[:, 1]


# -----------------------------
# Plot the groups
# -----------------------------
plt.figure(figsize=(8, 6))

for group_name in df["GMM_Group"].dropna().unique():
    subset = df[df["GMM_Group"] == group_name]

    plt.scatter(
        subset["PCA1"],
        subset["PCA2"],
        label=group_name,
        alpha=0.7
    )

plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.title(f"GMM Classification Using {anchor_A_clean} and {anchor_B_clean} Anchors")
plt.legend()
plt.tight_layout()

pca_plot_file = OUTPUT_DIR / f"{OUTPUT_PREFIX}_gmm_anchor_pca_plot.png"
plt.savefig(pca_plot_file, dpi=300)
plt.show()
print(f"Saved: {pca_plot_file}")


In [ ]:
# -----------------------------
# Well-level group summary
# -----------------------------
# Generalized summary columns: use whichever of these exist in the current dataset.
summary_group_cols = ["WELL_CLEAN"]
for possible_col in [temperature_col, time_col]:
    if possible_col and possible_col in df.columns and possible_col not in summary_group_cols:
        summary_group_cols.append(possible_col)

well_summary = (
    df
    .groupby(summary_group_cols, dropna=False)
    .agg(
        Most_Common_Group=("GMM_Group", lambda x: x.value_counts().index[0]),
        Mean_Probability_A_like=(f"Probability_{anchor_A_clean}_like", "mean"),
        Mean_Probability_B_like=(f"Probability_{anchor_B_clean}_like", "mean"),
        Number_of_Rows=("GMM_Group", "count")
    )
    .reset_index()
)

well_summary_file = OUTPUT_DIR / f"{OUTPUT_PREFIX}_gmm_well_level_group_summary.csv"
well_summary.to_csv(well_summary_file, index=False)
print(f"Saved: {well_summary_file}")
well_summary.head()


In [ ]:
# -----------------------------
# Row-level group list
# -----------------------------
# Keep the useful row-identifying columns that exist in this dataset.
row_id_candidates = [
    group_col,
    "WELL_CLEAN",
    temperature_col,
    time_col,
    "FOV",
    "T",
]
row_level_columns = []
for col in row_id_candidates + [
    "GMM_Group",
    f"Probability_{anchor_A_clean}_like",
    f"Probability_{anchor_B_clean}_like",
]:
    if col and col in df.columns and col not in row_level_columns:
        row_level_columns.append(col)

row_level_list = df[row_level_columns].copy()

row_level_file = OUTPUT_DIR / f"{OUTPUT_PREFIX}_gmm_row_level_group_list.csv"
row_level_list.to_csv(row_level_file, index=False)
print(f"Saved: {row_level_file}")
row_level_list.head()


In [ ]:
# -----------------------------
# Bar chart of group counts
# -----------------------------
group_counts = df["GMM_Group"].value_counts()

plt.figure(figsize=(7, 5))
group_counts.plot(kind="bar")

plt.xlabel("GMM Group")
plt.ylabel("Number of Rows")
plt.title("Number of Rows Assigned to Each GMM Group")
plt.tight_layout()

group_counts_plot_file = OUTPUT_DIR / f"{OUTPUT_PREFIX}_gmm_group_counts.png"
plt.savefig(group_counts_plot_file, dpi=300)
plt.show()
print(f"Saved: {group_counts_plot_file}")


In [ ]:
# -----------------------------
# Bar chart of well-level assignments
# -----------------------------
well_group_counts = (
    df
    .groupby(["WELL_CLEAN", "GMM_Group"], dropna=False)
    .size()
    .reset_index(name="Count")
)

pivot_table = well_group_counts.pivot(
    index="WELL_CLEAN",
    columns="GMM_Group",
    values="Count"
).fillna(0)

pivot_table.plot(kind="bar", stacked=True, figsize=(12, 6))

plt.xlabel("Well")
plt.ylabel("Number of Rows")
plt.title("GMM Group Assignments by Well")
plt.tight_layout()

assignments_plot_file = OUTPUT_DIR / f"{OUTPUT_PREFIX}_gmm_assignments_by_well.png"
plt.savefig(assignments_plot_file, dpi=300)
plt.show()
print(f"Saved: {assignments_plot_file}")


In [ ]:
# -----------------------------
# Save result
# -----------------------------
results_file = OUTPUT_DIR / f"{OUTPUT_PREFIX}_simple_gmm_anchor_results.csv"
df.to_csv(results_file, index=False)

print("Done!")
print(f"Saved: {results_file}")
print()
print("Features used:")
for col in features:
    print(" -", col)
print()
print("Group counts:")
print(df["GMM_Group"].value_counts())
